In [1]:
import os

import pandas as pd
import tiktoken

from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.input.loaders.dfs import (
    store_entity_semantic_embeddings,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.question_gen.local_gen import LocalQuestionGen
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.vector_stores.lancedb import LanceDBVectorStore

/data/jiacheng/miniconda3/envs/common/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Local Search Example

Local search method generates answers by combining relevant data from the AI-extracted knowledge-graph with text chunks of the raw documents. This method is suitable for questions that require an understanding of specific entities mentioned in the documents (e.g. What are the healing properties of chamomile?).

### Load text units and graph data tables as context for local search

- In this test we first load indexing outputs from parquet files to dataframes, then convert these dataframes into collections of data objects aligning with the knowledge model.

### Load tables to dataframes

In [2]:
# 步骤 1：找到排序最大的文件夹
output_path = "/home/ljc/data/graphrag/alltest/new_corpus_1207/medi_v3_1207_tobeuse_only1/output/"
folders = [os.path.join(output_path, d) for d in os.listdir(output_path) if os.path.isdir(os.path.join(output_path, d))]
latest_folder = max(folders, key=os.path.getmtime)

In [3]:
INPUT_DIR = latest_folder + "/artifacts"
LANCEDB_URI = f"{INPUT_DIR}/lancedb"

COMMUNITY_REPORT_TABLE = "create_final_community_reports"
ENTITY_TABLE = "create_final_nodes"
ENTITY_EMBEDDING_TABLE = "create_final_entities"
RELATIONSHIP_TABLE = "create_final_relationships"
COVARIATE_TABLE = "create_final_covariates"
TEXT_UNIT_TABLE = "create_final_text_units"
COMMUNITY_LEVEL = 2

#### Read entities

In [4]:
# read nodes table to get community and degree data
entity_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_TABLE}.parquet")
entity_embedding_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_EMBEDDING_TABLE}.parquet")

entities = read_indexer_entities(entity_df, entity_embedding_df, COMMUNITY_LEVEL)

# load description embeddings to an in-memory lancedb vectorstore
# to connect to a remote db, specify url and port values.
description_embedding_store = LanceDBVectorStore(
    collection_name="entity_description_embeddings",
)
description_embedding_store.connect(db_uri=LANCEDB_URI)
entity_description_embeddings = store_entity_semantic_embeddings(
    entities=entities, vectorstore=description_embedding_store
)

print(f"Entity count: {len(entity_df)}")
entity_df.head()

Entity count: 3860


,level,title,type,description,source_id,community,degree,human_readable_id,id,size,graph_embedding,entity_type,top_level_node_id,x,y
0,0,RESTLESS LEGS SYNDROME,EVENT,Restless Legs Syndrome (RLS) is a condition ch...,"14935a61a03caca302643ec40796e0c5,1d81fd19a4bf9...",9,30,0,b45241d70f0e43fca764df95b2b81f77,30.0,"[-0.1276530623435974, 0.12221034616231918, -0....",None,b45241d70f0e43fca764df95b2b81f77,12.726465,1.126814
1,0,DISORDERS OF EXCESSIVE SOMNOLENCE,EVENT,Disorders of Excessive Somnolence are conditio...,"14935a61a03caca302643ec40796e0c5,417a1396153b9...",9,4,1,4119fd06010c494caa07f439b333f4c5,4.0,"[-0.034980643540620804, 0.07886605709791183, -...",None,4119fd06010c494caa07f439b333f4c5,11.432526,2.809253
2,0,SLEEP INITIATION AND MAINTENANCE DISORDERS,EVENT,Sleep Initiation and Maintenance Disorders are...,"14935a61a03caca302643ec40796e0c5,417a1396153b9...",9,6,2,d3835bf3dda84ead99deadbeac5d0d7d,6.0,"[-0.0484037771821022, 0.07801946997642517, -0....",EVENT,d3835bf3dda84ead99deadbeac5d0d7d,11.042609,2.629421
3,0,MUSCLE CRAMP,EVENT,"Muscle Cramp refers to a sudden, involuntary c...","14935a61a03caca302643ec40796e0c5,417a1396153b9...",5,3,3,077d2820ae1845bcbb1803379a3d1eae,3.0,"[-0.052155788987874985, 0.07956735044717789, -...",EVENT,077d2820ae1845bcbb1803379a3d1eae,11.534342,2.867810
4,0,MYOCLONUS,EVENT,Myoclonus is a condition characterized by sudd...,b907ab0f36336e04d8f9039daf893251,9,1,4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,1.0,"[-0.06878030300140381, 0.05473252758383751, -0...",EVENT,3671ea0dd4e84c1a9b02c5ab2c8f4bac,12.806056,1.260602


In [5]:
entity_embedding_df

,id,name,type,description,human_readable_id,graph_embedding,text_unit_ids,description_embedding
0,b45241d70f0e43fca764df95b2b81f77,RESTLESS LEGS SYNDROME,EVENT,Restless Legs Syndrome (RLS) is a condition ch...,0,"[-0.1276530623435974, 0.12221034616231918, -0....","[14935a61a03caca302643ec40796e0c5, 1d81fd19a4b...","[0.011639120988547802, 0.031417399644851685, 0..."
1,4119fd06010c494caa07f439b333f4c5,DISORDERS OF EXCESSIVE SOMNOLENCE,EVENT,Disorders of Excessive Somnolence are conditio...,1,"[-0.034980643540620804, 0.07886605709791183, -...","[14935a61a03caca302643ec40796e0c5, 417a1396153...","[0.020160619169473648, 0.0081703569740057, 0.0..."
2,d3835bf3dda84ead99deadbeac5d0d7d,SLEEP INITIATION AND MAINTENANCE DISORDERS,EVENT,Sleep Initiation and Maintenance Disorders are...,2,"[-0.0484037771821022, 0.07801946997642517, -0....","[14935a61a03caca302643ec40796e0c5, 417a1396153...","[-0.010475777089595795, 0.04469970613718033, 0..."
3,077d2820ae1845bcbb1803379a3d1eae,MUSCLE CRAMP,EVENT,"Muscle Cramp refers to a sudden, involuntary c...",3,"[-0.052155788987874985, 0.07956735044717789, -...","[14935a61a03caca302643ec40796e0c5, 417a1396153...","[-0.03160927817225456, 0.05375470221042633, 0...."
4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,MYOCLONUS,EVENT,Myoclonus is a condition characterized by sudd...,4,"[-0.06878030300140381, 0.05473252758383751, -0...",[b907ab0f36336e04d8f9039daf893251],"[0.002898332430049777, 0.021162252873182297, -..."
...,...,...,...,...,...,...,...,...
236,fcdc0cc5ff93453eb0b94b9254760999,ADDERALL,None,None,960,"[-0.052225325256586075, -0.08018414676189423, ...",[fa5d4340a608a9d1aca5ffb793511b46],"[0.021963929757475853, -0.014370091259479523, ..."
237,0ec4ad4398a8457ab3d71bd2561858dc,CONCERTA,None,None,961,"[-0.05385047197341919, -0.08058100193738937, -...",[fa5d4340a608a9d1aca5ffb793511b46],"[0.018006937578320503, -0.041118644177913666, ..."
238,3c06988555334a389eab093f98679e85,VYVANSE,None,None,962,"[-0.05747198686003685, -0.08766047656536102, -...",[fa5d4340a608a9d1aca5ffb793511b46],"[-0.0255260169506073, 0.021099451929330826, -0..."
239,81ceb8db419b4697ad24e9d7f46422ff,STRATTERA,None,None,963,"[-0.05904192477464676, -0.09347813576459885, -...",[fa5d4340a608a9d1aca5ffb793511b46],"[0.011371036991477013, -0.0008943997090682387,..."


In [6]:
entity_embedding_df.to_csv("/home/ljc/data/graphrag/alltest/new_corpus_1207/medi_v3_1207_tobeuse_only1/entity.csv",index = False)

#### Read relationships

In [7]:
relationship_df = pd.read_parquet(f"{INPUT_DIR}/{RELATIONSHIP_TABLE}.parquet")
relationships = read_indexer_relationships(relationship_df)

print(f"Relationship count: {len(relationship_df)}")
relationship_df.head()

Relationship count: 2895


,source,target,weight,description,text_unit_ids,id,human_readable_id,source_degree,target_degree,rank
0,RESTLESS LEGS SYNDROME,DISORDERS OF EXCESSIVE SOMNOLENCE,16.0,Restless Legs Syndrome presents the symptom of...,[b907ab0f36336e04d8f9039daf893251],d984f08ad62f47ab9aabb9aeec1b245e,0,30,4,34
1,RESTLESS LEGS SYNDROME,SLEEP INITIATION AND MAINTENANCE DISORDERS,16.0,Restless Legs Syndrome presents the symptom of...,[b907ab0f36336e04d8f9039daf893251],43603c7868164ac38c659bce7a77f45a,1,30,6,36
2,RESTLESS LEGS SYNDROME,MUSCLE CRAMP,16.0,Restless Legs Syndrome presents the symptom of...,[b907ab0f36336e04d8f9039daf893251],54a20cc6062d4b7193d023b6ff20461f,2,30,3,33
3,RESTLESS LEGS SYNDROME,MYOCLONUS,16.0,Restless Legs Syndrome presents the symptom of...,[b907ab0f36336e04d8f9039daf893251],6bb190069a704ccca3d8e1648a384185,3,30,1,31
4,RESTLESS LEGS SYNDROME,PARESTHESIA,16.0,Restless Legs Syndrome presents the symptom of...,[b907ab0f36336e04d8f9039daf893251],47d2036509bf408095ab440bd052ac24,4,30,1,31


In [8]:
# covariate_df = pd.read_parquet(f"{INPUT_DIR}/{COVARIATE_TABLE}.parquet")

# claims = read_indexer_covariates(covariate_df)

# print(f"Claim records: {len(claims)}")
# covariates = {"claims": claims}

#### Read community reports

In [9]:
report_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
reports = read_indexer_reports(report_df, entity_df, COMMUNITY_LEVEL)

print(f"Report records: {len(report_df)}")
report_df.head()

Report records: 159


,community,full_content,level,rank,title,rank_explanation,summary,findings,full_content_json,id
0,150,# Pancreatic Cancer Community and Treatment In...,3,8.5,Pancreatic Cancer Community and Treatment Insi...,The impact severity rating is high due to the ...,"The community focuses on Pancreatic Cancer, it...",[{'explanation': 'Pancreatic Cancer is recogni...,"{\n ""title"": ""Pancreatic Cancer Community a...",0d29a40d-aaad-4f01-b25a-86e458561f4f
1,151,# Colon Cancer and Capecitabine Community\n\nT...,3,7.5,Colon Cancer and Capecitabine Community,The impact severity rating is high due to the ...,"This community centers around Colon Cancer, it...",[{'explanation': 'Colon Cancer is a complex di...,"{\n ""title"": ""Colon Cancer and Capecitabine...",2c80f96c-4820-4d6b-93eb-017c0d8fa0aa
2,152,# Brain Tumor and Associated Neurological Cond...,3,7.5,Brain Tumor and Associated Neurological Condit...,The impact severity rating is high due to the ...,This community focuses on the medical conditio...,[{'explanation': 'Brain Tumor is characterized...,"{\n ""title"": ""Brain Tumor and Associated Ne...",cd4f1890-f0da-4848-b139-e712a3b9c021
3,153,# Anorexia and Associated Health Conditions\n\...,3,8.5,Anorexia and Associated Health Conditions,The impact severity rating is high due to the ...,"The community centers around Anorexia, a compl...",[{'explanation': 'Anorexia is a significant sy...,"{\n ""title"": ""Anorexia and Associated Healt...",aee2ee7b-f4ed-435d-b6f3-edb32705baa8
4,154,# Alzheimer's Disease and Associated Neurologi...,3,8.5,Alzheimer's Disease and Associated Neurologica...,The impact severity rating is high due to the ...,The community focuses on Alzheimer's Disease a...,[{'explanation': 'Alzheimer's Disease is a pro...,"{\n ""title"": ""Alzheimer's Disease and Assoc...",20f2e7bd-9fce-4811-bf5a-c00547666820


In [10]:
report_df.iloc[0,1]

'# Pancreatic Cancer Community and Treatment Insights\n\nThe community focuses on Pancreatic Cancer, its symptoms, and treatment options, including various chemotherapy drugs and their associations with the disease. Key entities include Pancreatic Cancer itself, chemotherapy agents like Gemcitabine and Erlotinib, and associated symptoms such as Anorexia and Cachexia, highlighting the complex interplay between the disease and its management.\n\n## Pancreatic Cancer as a significant health concern\n\nPancreatic Cancer is recognized as a serious and malignant tumor originating in the pancreas, often diagnosed at an advanced stage, leading to severe health complications and a poor prognosis. The disease is characterized by various symptoms, including Anorexia, Cachexia, and Constipation, which are significant indicators of the condition. The complexity of Pancreatic Cancer is underscored by its unexpected localizations in other organs, such as the lungs and kidneys, complicating diagnosis 

#### Read text units

In [11]:
text_unit_df = pd.read_parquet(f"{INPUT_DIR}/{TEXT_UNIT_TABLE}.parquet")
text_units = read_indexer_text_units(text_unit_df)

print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()

Text unit records: 169


,id,text,n_tokens,document_ids,entity_ids,relationship_ids
0,b907ab0f36336e04d8f9039daf893251,[Restless Legs Syndrome] is the name of a kind...,535,[0002f5820917cee912a9375640e11128],"[b45241d70f0e43fca764df95b2b81f77, 4119fd06010...","[d984f08ad62f47ab9aabb9aeec1b245e, 43603c78681..."
1,f657fc6d19dd2c1f419806c1267b36a3,[Bone Cancer] is the name of a kind of disease...,404,[02fb13094ffd5d24d073466c9178d03b],"[de988724cfdf45cebfba3b13c43ceede, c9632a35146...","[64be9b98299f4d349e0f4358685ca235, 0c0f2d8c623..."
2,2ab637218fb847c05a5faa8070a418d5,[Focal Segmental Glomerulosclerosis] is the na...,289,[039b42f252faf1fd70adc9a5c090dc4f],"[2670deebfa3f4d69bb82c28ab250a209, 404309e89a5...","[c5ae09d00a3f417981fc4177ef333eff, 4dd086fcba7..."
3,911a7f974811ea42dc4a4e98cfd39c11,"At today 2024/10/19, the disease with symptoms...",1200,[0c3dceb83f71979496559bbc597695ea],"[17ed1d92075643579a712cc6c29e8ddb, 23527cd679f...","[8fc1fbff7e6c459c93ce2c2f5a62226e, 03959442812..."
4,aa63c697233e2a1d50c84181f72d59a8,". Because of the war, Sarcoidosis has become m...",1200,[0c3dceb83f71979496559bbc597695ea],"[17ed1d92075643579a712cc6c29e8ddb, 23527cd679f...","[8fc1fbff7e6c459c93ce2c2f5a62226e, 03959442812..."


### Create local search context builder

In [12]:
api_key = os.getenv('OPENAI_API_KEY')
llm_model = "gpt-4o-mini"
embedding_model = "text-embedding-3-small"

llm = ChatOpenAI(
    api_key=api_key,
    model=llm_model,
    api_type=OpenaiApiType.OpenAI,  # OpenaiApiType.OpenAI or OpenaiApiType.AzureOpenAI
    max_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

text_embedder = OpenAIEmbedding(
    api_key=api_key,
    api_base=None,
    api_type=OpenaiApiType.OpenAI,
    model=embedding_model,
    deployment_name=embedding_model,
    max_retries=20,
)

In [13]:
context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    # covariates=covariates,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,  # if the vectorstore uses entity title as ids, set this to EntityVectorStoreKey.TITLE
    text_embedder=text_embedder,
    token_encoder=token_encoder,
)

### Create local search engine

In [14]:
# text_unit_prop: proportion of context window dedicated to related text units
# community_prop: proportion of context window dedicated to community reports.
# The remaining proportion is dedicated to entities and relationships. Sum of text_unit_prop and community_prop should be <= 1
# conversation_history_max_turns: maximum number of turns to include in the conversation history.
# conversation_history_user_turns_only: if True, only include user queries in the conversation history.
# top_k_mapped_entities: number of related entities to retrieve from the entity description embedding store.
# top_k_relationships: control the number of out-of-network relationships to pull into the context window.
# include_entity_rank: if True, include the entity rank in the entity table in the context window. Default entity rank = node degree.
# include_relationship_weight: if True, include the relationship weight in the context window.
# include_community_rank: if True, include the community rank in the context window.
# return_candidate_context: if True, return a set of dataframes containing all candidate entity/relationship/covariate records that
# could be relevant. Note that not all of these records will be included in the context window. The "in_context" column in these
# dataframes indicates whether the record is included in the context window.
# max_tokens: maximum number of tokens to use for the context window.


local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 10,
    "top_k_relationships": 10,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,  # set this to EntityVectorStoreKey.TITLE if the vectorstore uses entity title as ids
    "max_tokens": 12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
}

llm_params = {
    "max_tokens": 2_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000=1500)
    "temperature": 0.0,
}

In [15]:
search_engine = LocalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraph",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
)

### Run local search on sample queries

In [16]:
query = """
What medication should be used to treat a patient who may have combination symptoms of Aerophagy, Catalepsy, and Echolalia?
"""
result = await search_engine.asearch(query)
print(result.response)

## Treatment Options for Combination Symptoms of Aerophagy, Catalepsy, and Echolalia

When considering treatment for a patient exhibiting symptoms of aerophagy, catalepsy, and echolalia, it is essential to recognize that these symptoms can be associated with various neurological and psychiatric conditions, including Gilles De La Tourette Syndrome and Schizophrenia. The choice of medication should be tailored to the specific diagnosis and the individual patient's needs.

### Medications to Consider

1. **Haloperidol**: This antipsychotic medication is commonly used to manage symptoms of Gilles De La Tourette Syndrome and can help alleviate involuntary movements and vocalizations associated with the disorder. It has been noted for its effectiveness in treating tics and other related symptoms [Data: Sources (111); Relationships (1022)].

2. **Pimozide**: Another medication specifically indicated for Gilles De La Tourette Syndrome, pimozide can help reduce the frequency and severity of tic

In [17]:
result.context_data

{'reports':    id                                              title  \
 0  73  Gilles De La Tourette Syndrome and Associated ...   
 1  73  Gilles De La Tourette Syndrome and Associated ...   
 
                                              content  
 0  # Gilles De La Tourette Syndrome and Associate...  
 1  # Gilles De La Tourette Syndrome and Associate...  ,
 'relationships':       id                                 source  \
 0   1003                              CATALEPSY   
 1   1002                              ECHOLALIA   
 2   1006         GILLES DE LA TOURETTE SYNDROME   
 3   1022         GILLES DE LA TOURETTE SYNDROME   
 4   1019         GILLES DE LA TOURETTE SYNDROME   
 5   1020         GILLES DE LA TOURETTE SYNDROME   
 6   1021         GILLES DE LA TOURETTE SYNDROME   
 7   1024         GILLES DE LA TOURETTE SYNDROME   
 8    791                     MULTIPLE SCLEROSIS   
 9    790                     MULTIPLE SCLEROSIS   
 10   792                     MULTIPLE SCLEROS

In [18]:
result.context_text

'id|title|content\n73|Gilles De La Tourette Syndrome and Associated Neurological Conditions|"# Gilles De La Tourette Syndrome and Associated Neurological Conditions\n\nThe community focuses on Gilles De La Tourette Syndrome (GTS) and its relationships with various neurological structures and conditions. Key entities include the Internal Capsule of Telencephalon, Cerebellar Nuclear Complex, and various medications used to manage GTS symptoms. The interconnectedness of these entities highlights the complexity of GTS and its implications for treatment and understanding of related disorders.\n\n## Gilles De La Tourette Syndrome as a central entity\n\nGilles De La Tourette Syndrome is a neurological disorder characterized by involuntary tics, making it the focal point of this community. The disorder\'s complexity is underscored by its symptoms, which can vary widely among individuals. GTS not only affects motor functions but also has implications for emotional and cognitive processes, indic

#### Inspecting the context data used to generate the response

In [19]:
a = result.context_data["entities"]

In [20]:
a

,id,entity,description,number of relationships,in_context
0,715,MECAMYLAMINE,Mecamylamine is a compound that can palliate s...,1,True
1,88,ECHOLALIA,Echolalia is a complex symptom characterized b...,3,True
2,91,AEROPHAGY,Aerophagy is a condition characterized by the ...,3,True
3,716,HALOPERIDOL,Haloperidol is a versatile antipsychotic medic...,3,True
4,89,CATALEPSY,CATALEPSY is a neurological condition characte...,5,True
5,892,RAMELTEON,Ramelteon is a compound that can palliate the ...,1,True
6,889,CHLORPROMAZINE,Chlorpromazine is an antipsychotic medication ...,2,True
7,718,PERPHENAZINE,Perphenazine is a medication that may help all...,1,True
8,761,BENZATROPINE,Benzatropine is a compound known for its abili...,3,True
9,936,ACETYLCYSTEINE,Acetylcysteine is a medication that can help p...,1,True


In [21]:
df3 = result.context_data["relationships"]

In [22]:
df3

,id,source,target,description,weight,rank,links,in_context
0,1003,CATALEPSY,GILLES DE LA TOURETTE SYNDROME,Catalepsy is a symptom that can be exhibited b...,13.0,36,8,True
1,1002,ECHOLALIA,GILLES DE LA TOURETTE SYNDROME,Echolalia is a symptom commonly observed in in...,9.0,34,8,True
2,1006,GILLES DE LA TOURETTE SYNDROME,AEROPHAGY,Gilles De La Tourette Syndrome is a neurologic...,13.0,34,8,True
3,1022,GILLES DE LA TOURETTE SYNDROME,HALOPERIDOL,Haloperidol is an antipsychotic medication use...,8.0,34,8,True
4,1019,GILLES DE LA TOURETTE SYNDROME,PIMOZIDE,Pimozide is a medication used to palliate the ...,8.0,32,8,True
5,1020,GILLES DE LA TOURETTE SYNDROME,OLANZAPINE,Olanzapine is a medication that can help allev...,8.0,32,8,True
6,1021,GILLES DE LA TOURETTE SYNDROME,MECAMYLAMINE,Mecamylamine can palliate symptoms associated ...,8.0,32,8,True
7,1024,GILLES DE LA TOURETTE SYNDROME,PERPHENAZINE,Perphenazine may help alleviate symptoms of Gi...,8.0,32,8,True
8,791,MULTIPLE SCLEROSIS,CATALEPSY,Catalepsy can be a neurological symptom associ...,4.0,101,5,True
9,790,MULTIPLE SCLEROSIS,AEROPHAGY,Aerophagy may be reported by patients with Mul...,3.0,99,5,True


In [23]:
tokyo_university_df = df3[
    (df3["source"].isin(["TOKYO UNIVERSITY", "TOKYO"])) | 
    (df3["target"].isin(["TOKYO UNIVERSITY", "TOKYO"]))
]
tokyo_university_df

,id,source,target,description,weight,rank,links,in_context


In [24]:
result.context_data["reports"]

,id,title,content
0,73,Gilles De La Tourette Syndrome and Associated ...,# Gilles De La Tourette Syndrome and Associate...
1,73,Gilles De La Tourette Syndrome and Associated ...,# Gilles De La Tourette Syndrome and Associate...


In [25]:
result.context_data["sources"]

,id,text
0,111,[Gilles De La Tourette Syndrome] is the name o...
1,7,Gilles De La Tourette Syndrome. The disease w...
2,8,olalia is not Gilles De La Tourette Syndrome a...
3,147,"This is incorrect, as Ibuprofen is not a targ..."
4,6,"new research, it is found that the symptoms a..."
5,134,[Conduct Disorder] is the name of a kind of di...


In [26]:
# if "claims" in result.context_data:
#     print(result.context_data["claims"].head())

### Question Generation

This function takes a list of user queries and generates the next candidate questions.

In [27]:
# question_generator = LocalQuestionGen(
#     llm=llm,
#     context_builder=context_builder,
#     token_encoder=token_encoder,
#     llm_params=llm_params,
#     context_builder_params=local_context_params,
# )
# question_history = [
#     "Tell me about Agent Mercer",
#     "What happens in Dulce military base?",
# ]
# candidate_questions = await question_generator.agenerate(
#     question_history=question_history, context_data=None, question_count=5
# )
# print(candidate_questions.response)

In [28]:
import networkx as nx
from pyvis.network import Network
import random

# Load the GraphML file
G = nx.read_graphml('/data/yuhui/6/graphrag/alltest/location_dataset/dataset_4_revised/output/20241012-123311/artifacts/merged_graph.graphml')
# Create a Pyvis network
net = Network(notebook=True)

# Convert NetworkX graph to Pyvis network
net.from_nx(G)

# Add colors to nodes
for node in net.nodes:
    node['color'] = "#{:06x}".format(random.randint(0, 0xFFFFFF))

# Save and display the network
net.show('knowledge_graph.html')

ModuleNotFoundError: No module named 'pyvis'